# 2. Entrenamiento

Acá se eligen los parámetros y se entrena una población. Hay dos formas de entrenar:

| `POR_ETAPAS` | Qué hace |
|---|---|
| `True` | De lo simple a lo complejo: manzana → 2x2 → 3x3 → centro. La población pasa de una etapa a la siguiente cuando la mejor red llega en suficientes rutas de referencia. |
| `False` | Entrenamiento directo: un solo mapa, de principio a fin. |

Cada entrenamiento se guarda en `resultados/<fecha>_semilla<n>/` después de cada generación. Por etapas, cada etapa tiene su propia carpeta. Si se interrumpe la celda, se conserva la última generación completa.

Un entrenamiento completo, con 50 individuos, tarda horas. Para probar que todo anda alcanza con `POBLACION = 10` y pocas generaciones.

Para analizar el resultado se usa `3_evaluacion.ipynb`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # la carpeta del proyecto

from conduccion.aprendizaje.entrenamiento import entrenar, rutas_de_referencia
from conduccion.aprendizaje.etapas import Etapa, entrenar_por_etapas, etapas_santa_fe
from conduccion.aprendizaje.guardado import cargar
from conduccion.mapa.mapa import santa_fe
from conduccion.parametros import usar_parametros, volver_a_config
from conduccion.visualizacion.graficos import aplicar_estilo, graficar_entrenamiento

aplicar_estilo()

## 1. Cómo se entrena

Con la misma `SEMILLA`, dos variantes ven las mismas rutas y arrancan de la misma población. Para saber si una diferencia se repite, hay que probar varias semillas.

In [ ]:
POR_ETAPAS = True
SEMILLA = 0
POBLACION = 50

# Solo para el entrenamiento directo (POR_ETAPAS = False)
ZONA = "3x3"                # "2x2", "3x3" o "centro"
GENERACIONES = 300          # None = sigue hasta interrumpir la celda

# Solo por etapas: seguir desde una etapa guardada. None = empezar de cero.
# Con DESDE_ETAPA = None se sigue en la misma etapa donde se cortó, desde la generación siguiente:
# el resultado es el mismo que si no se hubiera cortado. Con un número (0 manzana, 1 2x2, 2 3x3,
# 3 centro), esa etapa empieza de cero con la población guardada.
CONTINUAR_DESDE = None
DESDE_ETAPA = None

### Las etapas

Las etapas se hacen en el orden del diccionario. Para saltear una, se borra su renglón.

| Valor | Qué es |
|---|---|
| `sigma` | σ de la mutación en esa etapa. Nunca aumenta de una etapa a la siguiente, para afinar sin desarmar lo aprendido. |
| `umbral` | pasa a la etapa siguiente cuando la mejor red llega en esta fracción de las rutas de referencia |
| `minimo_generaciones` | no pasa antes de esta cantidad de generaciones |
| `maximo_generaciones` | si no llega al umbral, pasa igual después de esta cantidad. La última etapa usa todas. |

Desde la segunda etapa, cada generación maneja además una ruta de una etapa anterior, para no olvidarla.

`ZONA_PRUEBA` es un barrio donde la red nunca entrena. Se mide una sola vez, al final.

In [ ]:
ETAPAS = {
    "manzana": {"sigma": 0.3,  "umbral": 1.0, "minimo_generaciones": 10, "maximo_generaciones": 150},
    "2x2":     {"sigma": 0.2,  "umbral": 0.9, "minimo_generaciones": 10, "maximo_generaciones": 250},
    "3x3":     {"sigma": 0.15, "umbral": 0.9, "minimo_generaciones": 10, "maximo_generaciones": 300},
    "centro":  {"sigma": 0.15, "umbral": 1.0, "minimo_generaciones": 10, "maximo_generaciones": 600},
}
ZONA_PRUEBA = "guadalupe"

## 2. Parámetros

Los valores de abajo son la configuración de base del informe, la misma que tiene `conduccion/config.py`.

| Parámetro | Qué es |
|---|---|
| `ACELERACION_LATERAL_MAXIMA` | límite de agarre en m/s²; `None` = sin límite |
| `PREMIO_TIEMPO` | λ, cuánto pesa llegar rápido; 0 = no se premia |
| `PENALIZACION_CARRIL` | c, cuánto descuenta el tiempo fuera del carril |
| `ELITE`, `TAMANO_TORNEO` | individuos que pasan sin cambios y tamaño del torneo |
| `PROBABILIDAD_CRUCE` | probabilidad de que un hijo mezcle a sus padres |
| `PROBABILIDAD_MUTACION` | en la mutación gaussiana, probabilidad de mutar cada gen |
| `NEURONAS_A_MUTAR` | en la mutación por neurona, cuántas neuronas cambian |
| `SIGMA` | σ del entrenamiento directo (por etapas se usa el de cada etapa) |
| `DESVIO_PESOS_INICIALES` | desvío de los pesos al azar de la primera generación |
| `RUTAS_POR_GENERACION` | rutas que maneja cada individuo en cada generación |
| `RUTAS_REFERENCIA` | rutas fijas con que se elige la mejor red |

Las variantes que compara el informe se obtienen cambiando un solo valor: `TIPO_CRUCE` (`"neurona"`, `"pesos"`, `"blx"`, `"ninguna"`), `TIPO_MUTACION` (`"gaussiana"`, `"neurona"`), `ACELERACION_LATERAL_MAXIMA` (`6.0` o `None`) y `POR_ETAPAS`.

In [ ]:
TIPO_CRUCE = "neurona"
TIPO_MUTACION = "gaussiana"

PARAMETROS = {
    "ACELERACION_LATERAL_MAXIMA": 6.0,
    "PREMIO_TIEMPO": 0.0,
    "PENALIZACION_CARRIL": 0.8,
    "ELITE": 2,
    "TAMANO_TORNEO": 3,
    "PROBABILIDAD_CRUCE": 0.8,
    "PROBABILIDAD_MUTACION": 0.05,
    "NEURONAS_A_MUTAR": 2,
    "SIGMA": 0.3,
    "DESVIO_PESOS_INICIALES": 0.1,
    "RUTAS_POR_GENERACION": 3,
    "RUTAS_REFERENCIA": 20,
}

## 3. Entrenar

Primero todo vuelve a los valores de `config.py`, y después se aplican los de arriba. Así no queda nada de una corrida anterior.

In [ ]:
volver_a_config()
usar_parametros(**PARAMETROS)

if POR_ETAPAS:
    etapas = etapas_santa_fe(SEMILLA, ETAPAS)
    mejor, identificadores = entrenar_por_etapas(etapas, semilla=SEMILLA, tamano_poblacion=POBLACION,
                                                 tipo_cruce=TIPO_CRUCE, tipo_mutacion=TIPO_MUTACION,
                                                 continuar_desde=CONTINUAR_DESDE, desde_etapa=DESDE_ETAPA)
    for identificador in identificadores:
        _, historia, datos = cargar(identificador)
        graficar_entrenamiento(historia, "Etapa " + datos["parametros"]["nombre_etapa"])
else:
    mapa = santa_fe(ZONA)
    print("Se entrena en:", mapa.nombre)
    mejor, historia = entrenar(mapa, semilla=SEMILLA, generaciones=GENERACIONES, tamano_poblacion=POBLACION,
                               tipo_cruce=TIPO_CRUCE, tipo_mutacion=TIPO_MUTACION)
    graficar_entrenamiento(historia, "Entrenamiento directo en " + mapa.nombre)

## 4. Prueba en un barrio que la red nunca vio

La mejor red final se mide en las 20 rutas de referencia de cada etapa y en las de `ZONA_PRUEBA`. Si en el barrio de prueba llega casi igual que en los de entrenamiento, la red generaliza.

Para una evaluación más amplia, en los 58 recortes de la ciudad, se usa `3_evaluacion.ipynb`.

In [ ]:
mapa_prueba = santa_fe(ZONA_PRUEBA)
prueba = Etapa(ZONA_PRUEBA, mapa_prueba, None, rutas_de_referencia(mapa_prueba, SEMILLA), 0, 0, 0, 0)
etapas_a_medir = etapas_santa_fe(SEMILLA, ETAPAS) if POR_ETAPAS else []

for etapa in etapas_a_medir + [prueba]:
    nota, _, llegadas = etapa.medir(mejor, SEMILLA)
    uso = "prueba" if etapa is prueba else "entrenamiento"
    print(f"{etapa.nombre:10s} {uso:13s} nota {nota:.3f}   llega en {llegadas}/{len(etapa.referencia)}")